# Notebook 04 — Durable Execution

This notebook explains Temporal's durable execution model — how Workflows survive failures through event history replay.

## What You Will Learn
- What event history is and how it works
- How replay reconstructs Workflow state
- Durable timers
- Why Workflow determinism matters

## Event History

Every Workflow execution produces an event history:

```
1. WorkflowExecutionStarted     ← Workflow received input
2. WorkflowTaskScheduled        ← Cluster dispatched task
3. WorkflowTaskStarted          ← Worker picked up task
4. WorkflowTaskCompleted        ← Worker completed task
5. ActivityTaskScheduled        ← Activity dispatched
6. ActivityTaskStarted          ← Activity started
7. ActivityTaskCompleted        ← Activity returned result
8. TimerStarted                 ← asyncio.sleep() called
9. TimerFired                   ← Timer completed
10. ActivityTaskScheduled       ← Next Activity dispatched
11. ActivityTaskCompleted       ← Next Activity completed
12. WorkflowExecutionCompleted  ← Workflow finished
```

## How Replay Works

When a Worker resumes a Workflow after a crash:

1. Temporal sends the full event history to the Worker
2. Worker replays the Workflow code from the beginning
3. Activity calls use **cached results** from history (not re-executed)
4. Timers are **skipped** (already recorded)
5. Once replay catches up, normal execution continues

```
Original:  Activity1 → Timer → Activity2
Replay:    [cached]  → [skip] → Activity2 (executes normally)
```

In [ ]:
# Simulate event history as a data structure
event_history = [
    {"id": 1, "type": "WorkflowExecutionStarted", "input": "Pierre, fr"},
    {"id": 2, "type": "WorkflowTaskScheduled"},
    {"id": 3, "type": "WorkflowTaskStarted"},
    {"id": 4, "type": "WorkflowTaskCompleted"},
    {"id": 5, "type": "ActivityTaskScheduled", "activity": "translate_term(hello, fr)"},
    {"id": 6, "type": "ActivityTaskStarted"},
    {"id": 7, "type": "ActivityTaskCompleted", "result": "Bonjour"},
    {"id": 8, "type": "TimerStarted", "duration": "10s"},
    {"id": 9, "type": "TimerFired"},
    {"id": 10, "type": "ActivityTaskScheduled", "activity": "translate_term(goodbye, fr)"},
    {"id": 11, "type": "ActivityTaskStarted"},
    {"id": 12, "type": "ActivityTaskCompleted", "result": "Au revoir"},
    {"id": 13, "type": "WorkflowExecutionCompleted"},
]

print("Event History:")
print("-" * 60)
for event in event_history:
    details = ""
    if "input" in event:
        details = f" → input: {event['input']}"
    elif "activity" in event:
        details = f" → {event['activity']}"
    elif "result" in event:
        details = f" → result: {event['result']}"
    elif "duration" in event:
        details = f" → duration: {event['duration']}"
    print(f"  {event['id']:>2}. {event['type']}{details}")

## Durable Timers

`asyncio.sleep()` in a Workflow becomes a durable timer:

```python
await asyncio.sleep(10)  # 10-second durable timer
```

- Recorded as `TimerStarted` / `TimerFired` in history
- Survives Worker crashes
- Can be hours, days, or months long

## Translation Workflow Example

```python
@workflow.defn
class TranslationWorkflow:
    @workflow.run
    async def run(self, input):
        # Activity 1: translate "hello"
        hello = await workflow.execute_activity_method(
            TranslationActivities.translate_term, hello_input,
            start_to_close_timeout=timedelta(seconds=5),
        )
        
        # Durable timer
        await asyncio.sleep(10)
        
        # Activity 2: translate "goodbye"
        goodbye = await workflow.execute_activity_method(
            TranslationActivities.translate_term, goodbye_input,
            start_to_close_timeout=timedelta(seconds=5),
        )
        
        return TranslationWorkflowOutput(
            hello_message=f"{hello.translation}, {input.name}",
            goodbye_message=f"{goodbye.translation}, {input.name}",
        )
```

## Why Determinism Matters

During replay, Workflow code must produce the **same sequence of commands**.

| ✅ Allowed | ❌ Not Allowed |
|-----------|---------------|
| `workflow.execute_activity()` | `requests.get()` |
| `asyncio.sleep()` | `time.sleep()` |
| `workflow.now()` | `datetime.now()` |
| `workflow.logger` | `random.random()` |

In [ ]:
# Demonstrate replay simulation
def simulate_replay(event_history: list) -> dict:
    """Simulate how Temporal replays a Workflow."""
    state = {"activities_replayed": 0, "timers_skipped": 0}
    
    for event in event_history:
        event_type = event["type"]
        
        if event_type == "ActivityTaskCompleted":
            state["activities_replayed"] += 1
            print(f"  Replay: Activity result '{event.get('result', '')}' from cache")
        elif event_type == "TimerFired":
            state["timers_skipped"] += 1
            print(f"  Replay: Timer skipped (already fired)")
        elif event_type == "WorkflowExecutionCompleted":
            print(f"  Replay: Workflow state fully restored")
    
    return state


print("Simulating Workflow replay:")
print("-" * 40)
result = simulate_replay(event_history)
print(f"\nActivities replayed from cache: {result['activities_replayed']}")
print(f"Timers skipped: {result['timers_skipped']}")

## CLI: Viewing Event History

```bash
# View event history for a workflow
temporal workflow show --workflow-id translation-tasks-example
```

Or use the Web UI at `http://localhost:8233`.

## Summary

- Temporal records every step as an **event** in the event history
- On failure, Workflows are **replayed** from history to restore state
- Activities use **cached results** during replay
- Timers are **skipped** during replay
- Workflow code must be **deterministic** for replay to work

**Next:** [05_testing_workflows.ipynb](05_testing_workflows.ipynb)